In [1]:
!pip install torch torchvision transformers pandas scikit-learn pillow

Defaulting to user installation because normal site-packages is not writeable
  Using cached torch-2.10.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
  Using cached pandas-3.0.1-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached pillow-12.1.1-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.2.28-cp312-cp312-win_amd64.whl.metadata (41 kB)
  Using cac

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

Transformer for Steel

In [ ]:
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import accuracy_score
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
image_folder = r"C:\Users\rp122\Documents\8th sem\Datasets\steel-defect-detection\train_images"
csv_path = r"C:\Users\rp122\Documents\8th sem\Datasets\steel-defect-detection\train_full.csv"

# Load CSV
df = pd.read_csv(csv_path)

# Keep only rows with defects (EncodedPixels not null)
df = df[df["EncodedPixels"].notnull()]

# If multiple rows per image → keep first class
df = df.groupby("ImageId").first().reset_index()

class SteelDataset(Dataset):
    def __init__(self, dataframe, image_folder, processor):
        self.df = dataframe
        self.image_folder = image_folder
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["ImageId"]
        label = int(self.df.iloc[idx]["ClassId"]) - 1  # make 0-based

        img_path = os.path.join(self.image_folder, img_name)
        image = Image.open(img_path).convert("RGB")

        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze()

        return pixel_values, torch.tensor(label)

# Load processor
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

dataset = SteelDataset(df, image_folder, processor)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# Load model
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=4,
    ignore_mismatched_sizes=True
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

# Training
epochs = 5
model.train()

for epoch in range(epochs):
    all_preds = []
    all_labels = []
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {acc:.4f}")

c:\Users\rp122\Documents\8th sem\code\sugar\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 236.74it/s, Materializing param=vit.layernorm.weight]                                 
ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([4, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([4])          

Notes:
- MISMATCH	:ckpt weights w

In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score
from transformers import CLIPProcessor, CLIPModel

# Paths
image_folder = r"C:\Users\rp122\Documents\8th sem\Datasets\steel-defect-detection\train_images"
csv_path = r"C:\Users\rp122\Documents\8th sem\Datasets\steel-defect-detection\train.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP model
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

# Load CSV
df = pd.read_csv(csv_path)

# Keep rows with defects
df = df[df["EncodedPixels"].notnull()]

# Keep one label per image
df = df.groupby("ImageId").first().reset_index()

# Define class prompts
labels = [
    "steel surface defect class 1",
    "steel surface defect class 2",
    "steel surface defect class 3",
    "steel surface defect class 4"
]

predictions = []
true_labels = []

for _, row in df.iterrows():
    
    img_name = row["ImageId"]
    true_label = int(row["ClassId"]) - 1
    
    img_path = os.path.join(image_folder, img_name)

    try:
        image = Image.open(img_path).convert("RGB")
    except:
        continue

    inputs = processor(
        text=labels,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=1)

    pred = torch.argmax(probs).item()

    predictions.append(pred)
    true_labels.append(true_label)

# Calculate accuracy
accuracy = accuracy_score(true_labels, predictions)

print("Accuracy:", accuracy)

C:\Users\rp122\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 590/590 [00:01<00:00, 338.86it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the 